In [ ]:
# ============================================
# 🧠 HumanEval Evaluation for CodeGemma-2B
# ============================================

# Step 0: Install dependencies (only first run)
!pip install -q transformers datasets torch tqdm pandas huggingface_hub

# Step 1: Imports
import torch, math, time, subprocess, sys, tempfile, os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm
import pandas as pd
from huggingface_hub import login

# ============================================
# Step 2: Authenticate (SAFE)
# ============================================
# You will be prompted to paste your Hugging Face token securely.

# ============================================
# Step 3: Configuration
# ============================================
MODEL_NAME = "google/codegemma-2b"
DATASET_NAME = "openai_humaneval"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2
NUM_SAMPLES = 5
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.8
TOP_P = 0.95
EVAL_TIMEOUT = 2  # seconds per test case

print(f"Model: {MODEL_NAME}\nDataset: {DATASET_NAME}\nDevice: {DEVICE}")

# ============================================
# Step 4: Load model and tokenizer
# ============================================
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
print("✅ Model loaded successfully.")

# ============================================
# Step 5: Load dataset (HumanEval)
# ============================================
print("Loading HumanEval dataset...")
dataset = load_dataset(DATASET_NAME)["test"]
entries = list(dataset)
print(f"✅ Loaded {len(entries)} tasks.")

# ============================================
# Step 6: Helper functions
# ============================================
def pass_at_k(n:int, c:int, k:int) -> float:
    """Standard pass@k metric."""
    if c == 0: return 0.0
    if k > n: k = n
    try:
        return 1.0 - math.comb(n-c, k)/math.comb(n, k)
    except ValueError:
        return 0.0

def safe_eval_subprocess(code:str, test:str, timeout:int=EVAL_TIMEOUT) -> bool:
    """Run generated code + test in subprocess safely."""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as tf:
        tf.write(code + "\n" + test)
        fname = tf.name
    try:
        proc = subprocess.run([sys.executable, fname],
                              stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE,
                              timeout=timeout)
        return proc.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        try: os.remove(fname)
        except: pass

# ============================================
# Step 7: Evaluation Loop
# ============================================
results = []
num_tasks = len(entries)

for i in tqdm(range(0, num_tasks, BATCH_SIZE), desc="Batches"):
    batch = entries[i:i+BATCH_SIZE]
    prompts = [entry["prompt"] for entry in batch]
    tests = [entry["test"] for entry in batch]
    task_ids = [entry["task_id"] for entry in batch]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            num_return_sequences=NUM_SAMPLES,
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    # Process each task
    for idx, (prompt, test, task_id) in enumerate(zip(prompts, tests, task_ids)):
        completions = decoded[idx*NUM_SAMPLES:(idx+1)*NUM_SAMPLES]
        correct_count = 0
        for c in completions:
            if c.startswith(prompt):
                c = c[len(prompt):].lstrip("\n")
            full_code = prompt + "\n" + c
            if safe_eval_subprocess(full_code, test):
                correct_count += 1
        results.append({
            "task_id": task_id,
            "correct_count": correct_count,
            "pass@1": pass_at_k(NUM_SAMPLES, correct_count, 1),
            "pass@5": pass_at_k(NUM_SAMPLES, correct_count, 5),
            "pass@10": pass_at_k(NUM_SAMPLES, correct_count, 10)
        })

# ============================================
# Step 8: Save and summarize
# ============================================
df = pd.DataFrame(results)
df.to_csv(f"{MODEL_NAME.replace('/', '_')}_humaneval_results.csv", index=False)
summary = {
    "tasks": len(df),
    "avg_pass@1": df["pass@1"].mean(),
    "avg_pass@5": df["pass@5"].mean(),
    "avg_pass@10": df["pass@10"].mean()
}
print("✅ Evaluation completed successfully.")
print(summary)


Model: google/codegemma-2b
Dataset: openai_humaneval
Device: cuda
Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded successfully.
Loading HumanEval dataset...
✅ Loaded 164 tasks.


Batches: 100%|██████████| 82/82 [07:43<00:00,  5.65s/it]

✅ Evaluation completed successfully.
{'tasks': 164, 'avg_pass@1': np.float64(0.09390243902439023), 'avg_pass@5': np.float64(0.3597560975609756), 'avg_pass@10': np.float64(0.3597560975609756)}


In [ ]:
# === Colab-ready: HumanEval evaluation for deepseek-ai/deepseek-coder-1.3b-base ===

# 0) (Only first run) Install dependencies
!pip install -q transformers datasets accelerate torch tqdm pandas

# 1) Imports
import os, sys, math, time, tempfile, subprocess
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from tqdm import tqdm
import pandas as pd

# 2) Config - change these if you like
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"
DATASET_NAME = "openai_humaneval"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4           # prompts per batch
NUM_SAMPLES = 5          # number of completions per prompt (n for pass@k)
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.8
TOP_P = 0.95
EVAL_TIMEOUT = 4         # seconds per generated completion (test timeout)
SEED = 42

print(f"CONFIG: {MODEL_NAME} | DATASET={DATASET_NAME} | DEVICE={DEVICE} | NUM_SAMPLES={NUM_SAMPLES} | BATCH={BATCH_SIZE}")

# 3) Safe evaluation helper (isolated subprocess)
def eval_code_with_test(code_str: str, test_str: str, timeout: int = EVAL_TIMEOUT) -> bool:
    """
    Writes code+test to a temp file and runs it in a subprocess.
    Returns True if tests passed (exit code 0), False otherwise or on timeout.
    """
    combined = code_str + "\n\n" + test_str + "\n"
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as tf:
        tf.write(combined)
        fname = tf.name
    try:
        proc = subprocess.run([sys.executable, fname],
                              stdout=subprocess.PIPE,
                              stderr=subprocess.PIPE,
                              timeout=timeout)
        return proc.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        try:
            os.remove(fname)
        except:
            pass

# 4) pass@k calculator (HumanEval formula)
def pass_at_k(n:int, c:int, k:int) -> float:
    if c == 0:
        return 0.0
    if k > n:
        k = n
    try:
        return 1.0 - math.comb(n - c, k) / math.comb(n, k)
    except ValueError:
        return 0.0

# 5) Load HumanEval dataset
print("Loading HumanEval dataset...")
humaneval = load_dataset(DATASET_NAME)["test"]
print("HumanEval size:", len(humaneval))

# 6) Load tokenizer & model (FP16 if GPU available)
print(f"Loading model {MODEL_NAME} (this may take a minute)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Fix padding token if missing (many code models omit pad_token)
if tokenizer.pad_token is None:
    # set pad_token to eos_token (safe choice)
    try:
        tokenizer.add_special_tokens({"pad_token": tokenizer.eos_token})
    except Exception:
        tokenizer.pad_token = tokenizer.eos_token

# Load model
if DEVICE == "cuda":
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
# If we added tokens, resize embeddings
try:
    model.resize_token_embeddings(len(tokenizer))
except Exception:
    pass

model.eval()
generation_device = next(model.parameters()).device
print("Model loaded. Example param device:", generation_device)

# reproducible sampling
torch.manual_seed(SEED)

# 7) Batched generation + evaluation
def evaluate_model_on_humaneval(dataset, tokenizer, model,
                                batch_size=BATCH_SIZE, num_samples=NUM_SAMPLES,
                                max_new_tokens=MAX_NEW_TOKENS):
    results = []
    n_total = len(dataset)

    # prebuild list of (task_id, prompt, test)
    tasks = []
    for item in dataset:
        prompt = item.get("prompt") or item.get("code") or ""
        test = item.get("test") or item.get("ref") or ""
        task_id = item.get("task_id", None) or item.get("id", None) or len(tasks)
        tasks.append((task_id, prompt, test))

    for i in tqdm(range(0, n_total, batch_size), desc="Batches"):
        batch = tasks[i:i+batch_size]
        prompts = [p for (_id,p,t) in batch]
        tests = [t for (_id,p,t) in batch]
        task_ids = [_id for (_id,p,t) in batch]

        # tokenize with padding/truncation
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(generation_device)

        # generate num_return_sequences per prompt
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                max_new_tokens=max_new_tokens,
                num_return_sequences=num_samples,
                pad_token_id=tokenizer.pad_token_id
            )

        # decode outputs; length = batch_size * num_samples
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # evaluate each prompt's completions
        for idx_in_batch, (task_id, prompt, test_str) in enumerate(batch):
            start = idx_in_batch * num_samples
            end = start + num_samples
            completions = decoded[start:end]

            correct_count = 0
            for code_out in completions:
                out = code_out
                # strip prompt echo if present
                if out.startswith(prompt):
                    out = out[len(prompt):].lstrip("\n")
                full_code = prompt + "\n" + out

                passed = False
                if test_str and len(test_str.strip())>0:
                    passed = eval_code_with_test(full_code, test_str, timeout=EVAL_TIMEOUT)
                else:
                    # fallback heuristic if no test: presence of def/return
                    passed = ("def " in out) or ("return " in out)
                if passed:
                    correct_count += 1

            p1 = pass_at_k(num_samples, correct_count, 1)
            p5 = pass_at_k(num_samples, correct_count, 5)
            p10 = pass_at_k(num_samples, correct_count, 10)

            results.append({
                "task_id": task_id,
                "correct_count": correct_count,
                "num_samples": num_samples,
                "pass@1": p1,
                "pass@5": p5,
                "pass@10": p10
            })

        # small sleep to avoid throttle
        time.sleep(0.02)

    return results

# 8) Run evaluation
start_time = time.time()
results = evaluate_model_on_humaneval(humaneval, tokenizer, model,
                                      batch_size=BATCH_SIZE,
                                      num_samples=NUM_SAMPLES,
                                      max_new_tokens=MAX_NEW_TOKENS)
elapsed = time.time() - start_time
print(f"Evaluation done in {elapsed/60:.2f} minutes. {len(results)} tasks processed.")

# 9) Summarize and save
df = pd.DataFrame(results)
summary = {
    "tasks": len(df),
    "avg_pass@1": df["pass@1"].mean(),
    "avg_pass@5": df["pass@5"].mean(),
    "avg_pass@10": df["pass@10"].mean()
}
print("SUMMARY:", summary)

out_csv = f"{MODEL_NAME.replace('/', '_')}_humaneval_results.csv"
df.to_csv(out_csv, index=False)
print("Saved per-task results to", out_csv)


CONFIG: deepseek-ai/deepseek-coder-1.3b-base | DATASET=openai_humaneval | DEVICE=cuda | NUM_SAMPLES=5 | BATCH=4
Loading HumanEval dataset...
HumanEval size: 164
Loading model deepseek-ai/deepseek-coder-1.3b-base (this may take a minute)...


tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.69G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Model loaded. Example param device: cuda:0



Batches: 100%|██████████| 41/41 [05:13<00:00,  7.64s/it]

Evaluation done in 5.22 minutes. 164 tasks processed.
SUMMARY: {'tasks': 164, 'avg_pass@1': np.float64(0.424390243902439), 'avg_pass@5': np.float64(0.8963414634146342), 'avg_pass@10': np.float64(0.8963414634146342)}
Saved per-task results to deepseek-ai_deepseek-coder-1.3b-base_humaneval_results.csv


In [ ]:
# === Colab-ready HumanEvalPlus Evaluation for DeepSeek-Coder-1.3B-Base ===

!pip install -q transformers datasets torch tqdm pandas huggingface_hub accelerate

import os, sys, math, time, tempfile, subprocess
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from tqdm import tqdm
import pandas as pd

# ---- LOGIN Hugging Face ----
login()  # use your HF token when prompted

# ---- CONFIG ----
MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-base"
DATASET_NAME = "evalplus/humanevalplus"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 4
NUM_SAMPLES = 5
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.8
TOP_P = 0.95
EVAL_TIMEOUT = 4

# ---- SAFE SUBPROCESS EVAL ----
def eval_code_with_test(code: str, test: str, timeout=EVAL_TIMEOUT):
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as tf:
        tf.write(code + "\n" + test)
        fname = tf.name
    try:
        proc = subprocess.run([sys.executable, fname],
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                              timeout=timeout)
        return proc.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        try: os.remove(fname)
        except: pass

# ---- PASS@K ----
def pass_at_k(n, c, k):
    if c == 0: return 0.0
    if k > n: k = n
    try:
        return 1 - math.comb(n-c, k)/math.comb(n, k)
    except ValueError:
        return 0.0

# ---- LOAD MODEL ----
print(f"Loading {MODEL_NAME} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": tokenizer.eos_token})

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,
                                             torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
                                             device_map="auto" if DEVICE=="cuda" else None)
model.eval()
print("Model loaded.")

# ---- LOAD DATASET ----
print(f"Loading {DATASET_NAME}...")
dataset = load_dataset(DATASET_NAME)["test"]
entries = list(dataset)
print(f"✅ Loaded {len(entries)} tasks.")

# ---- EVALUATION LOOP ----
results = []

for i in tqdm(range(0, len(entries), BATCH_SIZE), desc="Batches"):
    batch = entries[i:i+BATCH_SIZE]
    prompts = [e["prompt"] for e in batch]
    tests = [e["test"] for e in batch]
    task_ids = [e.get("task_id", idx+i) for idx, e in enumerate(batch)]

    inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True).to(next(model.parameters()).device)
    with torch.inference_mode():
        outputs = model.generate(**inputs,
                                 max_new_tokens=MAX_NEW_TOKENS,
                                 do_sample=True,
                                 temperature=TEMPERATURE,
                                 top_p=TOP_P,
                                 num_return_sequences=NUM_SAMPLES,
                                 pad_token_id=tokenizer.pad_token_id)
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for idx, (prompt, test, task_id) in enumerate(zip(prompts, tests, task_ids)):
        completions = decoded[idx*NUM_SAMPLES:(idx+1)*NUM_SAMPLES]
        correct_count = 0
        for c in completions:
            if c.startswith(prompt):
                c = c[len(prompt):].lstrip("\n")
            full_code = prompt + "\n" + c
            if eval_code_with_test(full_code, test):
                correct_count += 1
        results.append({
            "task_id": task_id,
            "correct_count": correct_count,
            "pass@1": pass_at_k(NUM_SAMPLES, correct_count, 1),
            "pass@5": pass_at_k(NUM_SAMPLES, correct_count, 5),
            "pass@10": pass_at_k(NUM_SAMPLES, correct_count, 10)
        })

# ---- SAVE RESULTS ----
df = pd.DataFrame(results)
summary = {
    "tasks": len(df),
    "avg_pass@1": df["pass@1"].mean(),
    "avg_pass@5": df["pass@5"].mean(),
    "avg_pass@10": df["pass@10"].mean()
}
print("SUMMARY:", summary)
out_csv = f"{MODEL_NAME.replace('/', '_')}_humanevalplus_results.csv"
df.to_csv(out_csv, index=False)
print("Saved per-task results to", out_csv)


Loading deepseek-ai/deepseek-coder-1.3b-base on cuda...
Model loaded.
Loading evalplus/humanevalplus...


README.md:   0%|          | 0.00/551 [00:00<?, ?B/s]

data/test-00000-of-00001-5973903632b82d4(…):   0%|          | 0.00/2.90M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

✅ Loaded 164 tasks.


Batches: 100%|██████████| 41/41 [05:34<00:00,  8.16s/it]

SUMMARY: {'tasks': 164, 'avg_pass@1': np.float64(0.40365853658536577), 'avg_pass@5': np.float64(0.8780487804878049), 'avg_pass@10': np.float64(0.8780487804878049)}
Saved per-task results to deepseek-ai_deepseek-coder-1.3b-base_humanevalplus_results.csv


In [ ]:
# === Install dependencies ===
!pip install -q transformers datasets evaluate bert-score accelerate torch tqdm pandas sentencepiece

# === Imports ===
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from evaluate import load as load_metric
from huggingface_hub import login

# === Login to Hugging Face (paste your token here when prompted) ===
login()

# === Configuration ===
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "tasksource/planbench"
CONFIG_NAME = "task_1_plan_generation"
SPLIT = "train"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 160
TEMPERATURE = 0.8
TOP_P = 0.95
OUTPUT_CSV = f"{MODEL_NAME.replace('/', '_')}_{CONFIG_NAME}_BERTScore.csv"

print(f"\n🚀 Model: {MODEL_NAME}")
print(f"📘 Dataset: {DATASET_NAME}/{CONFIG_NAME}")
print(f"💻 Device: {DEVICE}\n")

# === Load dataset ===
dataset = load_dataset(DATASET_NAME, CONFIG_NAME, split=SPLIT)
print(f"✅ Loaded {len(dataset)} tasks.\n")
print("Columns:", dataset.column_names)
print("Sample:", dataset[0])

# === Prepare examples ===
examples = []
for ex in dataset:
    prompt = ex["query"]
    reference = ex["ground_truth_plan"]
    if prompt.strip() and reference.strip():
        examples.append({"prompt": prompt.strip(), "reference": reference.strip()})

print(f"✅ Filtered {len(examples)} valid examples with references.\n")
if len(examples) == 0:
    raise SystemExit("No valid examples found — check dataset fields.")

# === Load model & tokenizer ===
print("📥 Loading model and tokenizer ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": tokenizer.eos_token})

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None
)
model.eval()
print("✅ Model loaded successfully.\n")

# === Text generation function ===
def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(DEVICE)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if text.startswith(prompt):
        text = text[len(prompt):].lstrip()
    return text.strip()

# === Generate outputs for all examples ===
print("🧠 Generating model outputs ...\n")
preds, refs, prompts = [], [], []
for ex in tqdm(examples, desc="Generating"):
    gen = generate_text(ex["prompt"])
    preds.append(gen)
    refs.append(ex["reference"])
    prompts.append(ex["prompt"])

# === Compute BERTScore ===
print("\n📊 Computing BERTScore ...\n")
bertscore = load_metric("bertscore")
bs = bertscore.compute(predictions=preds, references=refs, lang="en")
mean_f1 = float(np.mean(bs["f1"]))
print(f"✅ Mean BERTScore F1: {mean_f1:.4f}\n")

# === Save results ===
df = pd.DataFrame({
    "prompt": prompts,
    "reference": refs,
    "prediction": preds,
    "bertscore_f1": bs["f1"]
})
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"✅ Results saved to {OUTPUT_CSV}")
print("🎯 Done.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00



🚀 Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
📘 Dataset: tasksource/planbench/task_1_plan_generation
💻 Device: cuda



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

task_1_plan_generation/train-00000-of-00(…):   0%|          | 0.00/1.03M [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Mean BERTScore F1: 0.8232

✅ Results saved to TinyLlama_TinyLlama-1.1B-Chat-v1.0_task_1_plan_generation_BERTScore.csv
🎯 Done.


Saved: clean_notebook.ipynb
